In [1]:
import pandas as pd

In [2]:
def load_dataset(filepath):
        df = pd.read_csv(filepath)
        # Display the raw table
        print(f"Dataset Loaded: {len(df)} job roles found. \n")
        display(df)

        job_roles = {}
        for _, row in df.iterrows():
                role = row["role"].strip()
                skills = row["skills"].strip().split()
                job_roles[role] = skills
        return job_roles    

In [3]:
dt = load_dataset("raw_skill.csv")

Dataset Loaded: 10 job roles found. 



,role,skills
0,Data Scientist,python sql machine_learning data_analysis stat...
1,Machine Learning Engineer,python tensorflow deep_learning machine_learni...
2,Backend Developer,python sql apis java databases algorithms
3,Frontend Developer,javascript html css react ui_design apis
4,DevOps Engineer,aws docker kubernetes linux ci_cd automation
5,Data Analyst,sql excel data_analysis statistics python visu...
6,AI Engineer,python machine_learning deep_learning tensorfl...
7,Cloud Architect,aws docker kubernetes automation networking linux
8,Cybersecurity Analyst,networking linux encryption security_protocols...
9,Full Stack Developer,python javascript html css sql apis react


In [4]:
print("\n Parsed Job Roles:")
for role, skills in dt.items():
    print(f"\n{role}")
    print(f"   → {', '.join(skills)}")


 Parsed Job Roles:

Data Scientist
   → python, sql, machine_learning, data_analysis, statistics, tensorflow

Machine Learning Engineer
   → python, tensorflow, deep_learning, machine_learning, algorithms, math

Backend Developer
   → python, sql, apis, java, databases, algorithms

Frontend Developer
   → javascript, html, css, react, ui_design, apis

DevOps Engineer
   → aws, docker, kubernetes, linux, ci_cd, automation

Data Analyst
   → sql, excel, data_analysis, statistics, python, visualization

AI Engineer
   → python, machine_learning, deep_learning, tensorflow, algorithms, math

Cloud Architect
   → aws, docker, kubernetes, automation, networking, linux

Cybersecurity Analyst
   → networking, linux, encryption, security_protocols, python, risk_analysis

Full Stack Developer
   → python, javascript, html, css, sql, apis, react


In [5]:
def build_vocabulary(job_roles):
    # Collect every skill from every role
    all_skills = []
    for role, skills in job_roles.items():
        for skill in skills:
            all_skills.append(skill)

    # Remove duplicates and sort
    vocabulary = sorted(set(all_skills))
    return vocabulary

def vectorize(skills, vocabulary):
    vector = []
    for word in vocabulary:
        if word in skills:
            vector.append(1)
        else:
            vector.append(0)
    return vector

def build_role_vectors(job_roles, vocabulary):
    role_vectors = {}
    for role, skills in job_roles.items():
        role_vectors[role] = vectorize(skills, vocabulary)
    return role_vectors


vocabulary   = build_vocabulary(dt)
role_vectors = build_role_vectors(dt, vocabulary)

print(f"Vocabulary size: {len(vocabulary)} unique skills\n")
print(f"Vocabulary: {vocabulary}\n")
print("Role Vectors:")
for role, vector in role_vectors.items():
    print(f"\n{role}")
    print(f"   → {vector}")

Vocabulary size: 29 unique skills

Vocabulary: ['algorithms', 'apis', 'automation', 'aws', 'ci_cd', 'css', 'data_analysis', 'databases', 'deep_learning', 'docker', 'encryption', 'excel', 'html', 'java', 'javascript', 'kubernetes', 'linux', 'machine_learning', 'math', 'networking', 'python', 'react', 'risk_analysis', 'security_protocols', 'sql', 'statistics', 'tensorflow', 'ui_design', 'visualization']

Role Vectors:

Data Scientist
   → [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0]

Machine Learning Engineer
   → [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0]

Backend Developer
   → [1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0]

Frontend Developer
   → [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]

DevOps Engineer
   → [0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Data Analyst
  

In [6]:
#TF-IDF Weighting

def calculate_idf(vocabulary, job_roles):
    """Calculate IDF weight for each skill"""
    total_roles = len(job_roles)
    idf_weights = {}
    
    for skill in vocabulary:
        # Count how many roles have this skill
        roles_with_skill = 0
        for role, skills in job_roles.items():
            if skill in skills:
                roles_with_skill += 1
        
        idf = math.log(total_roles / roles_with_skill)
        idf_weights[skill] = idf
    
    return idf_weights

def create_weighted_vectors(job_roles, vocabulary, idf_weights):
    """Convert each role into a weighted TF-IDF vector"""
    weighted_vectors = {}
    
    for role, skills in job_roles.items():
        weighted_vector = []
        for word in vocabulary:
            if word in skills:
                # Skill present: use its IDF weight
                weighted_vector.append(idf_weights[word])
            else:
                # Skill absent: 0
                weighted_vector.append(0.0)
        
        weighted_vectors[role] = weighted_vector
    
    return weighted_vectors

import math

idf_weights      = calculate_idf(vocabulary, dt)
weighted_vectors = create_weighted_vectors(dt, vocabulary, idf_weights)

print(f"TF-IDF Weights Calculated\n")
print(" Top 10 Rarest Skills (highest IDF weights):")
sorted_skills = sorted(idf_weights.items(), key=lambda x: x[1], reverse=True)
for skill, idf in sorted_skills[:10]:
    print(f"   {skill:20} → IDF = {idf:.3f}")

print("\n Weighted Vectors (sample):")
for role in list(weighted_vectors.keys())[:3]:
    print(f"\n{role}")
    print(f"   → {[round(x, 3) for x in weighted_vectors[role]]}")

TF-IDF Weights Calculated

 Top 10 Rarest Skills (highest IDF weights):
   ci_cd                → IDF = 2.303
   databases            → IDF = 2.303
   encryption           → IDF = 2.303
   excel                → IDF = 2.303
   java                 → IDF = 2.303
   risk_analysis        → IDF = 2.303
   security_protocols   → IDF = 2.303
   ui_design            → IDF = 2.303
   visualization        → IDF = 2.303
   automation           → IDF = 1.609

 Weighted Vectors (sample):

Data Scientist
   → [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.609, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.204, 0.0, 0.0, 0.357, 0.0, 0.0, 0.0, 0.916, 1.609, 1.204, 0.0, 0.0]

Machine Learning Engineer
   → [1.204, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.609, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.204, 1.609, 0.0, 0.357, 0.0, 0.0, 0.0, 0.0, 0.0, 1.204, 0.0, 0.0]

Backend Developer
   → [1.204, 1.204, 0.0, 0.0, 0.0, 0.0, 0.0, 2.303, 0.0, 0.0, 0.0, 0.0, 0.0, 2.303, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.357, 0.0, 0.0, 

In [7]:
#Cosine Similarity Engine

def cosine_similarity(vector_a, vector_b):
    """Calculate cosine similarity between two vectors"""
    import math
    
    # Dot product
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    
    # Magnitudes
    magnitude_a = math.sqrt(sum(a**2 for a in vector_a))
    magnitude_b = math.sqrt(sum(b**2 for b in vector_b))
    
    # Avoid division by zero
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    
    # Cosine similarity
    return dot_product / (magnitude_a * magnitude_b)

def find_recommendations(user_skills, job_roles, vocabulary, idf_weights):
    """Find top 3 job recommendations for a user"""
    
    # Convert user skills to weighted vector
    user_vector = []
    for word in vocabulary:
        if word in user_skills:
            user_vector.append(idf_weights[word])
        else:
            user_vector.append(0.0)
    
    # Calculate similarity with each job role
    scores = {}
    for role, role_vector in weighted_vectors.items():
        similarity = cosine_similarity(user_vector, role_vector)
        scores[role] = similarity
    
    # Sort by similarity (descending)
    ranked_jobs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    return ranked_jobs

# Test the engine
print("🔧 Recommendation Engine Ready!\n")

🔧 Recommendation Engine Ready!



In [8]:
#Interactive Recommender

def recommend_job_roles(user_input_string):
    """Main function - takes user input and returns top 3 job recommendations"""
    
    # Parse user input
    user_skills = user_input_string.lower().strip().split()
    
    # Get recommendations
    ranked_jobs = find_recommendations(user_skills, dt, vocabulary, idf_weights)
    
    # Display results
    print(f"\n📊 You entered skills: {user_skills}\n")
    print("🎯 Top 3 Recommended Job Roles:\n")
    
    for i, (role, score) in enumerate(ranked_jobs[:3], 1):
        percentage = round(score * 100, 1)
        print(f"{i}. {role}")
        print(f"   Match Score: {percentage}%\n")

# Example usage
print("=" * 50)
print("TECH STACK RECOMMENDER")
print("=" * 50)

# Test Case 1
recommend_job_roles("python machine_learning tensorflow")

# Test Case 2
recommend_job_roles("javascript html css react")

# Test Case 3
recommend_job_roles("aws docker kubernetes linux")

TECH STACK RECOMMENDER

📊 You entered skills: ['python', 'machine_learning', 'tensorflow']

🎯 Top 3 Recommended Job Roles:

1. Data Scientist
   Match Score: 57.8%

2. Machine Learning Engineer
   Match Score: 56.0%

3. AI Engineer
   Match Score: 56.0%


📊 You entered skills: ['javascript', 'html', 'css', 'react']

🎯 Top 3 Recommended Job Roles:

1. Full Stack Developer
   Match Score: 90.0%

2. Frontend Developer
   Match Score: 77.8%

3. Data Scientist
   Match Score: 0.0%


📊 You entered skills: ['aws', 'docker', 'kubernetes', 'linux']

🎯 Top 3 Recommended Job Roles:

1. Cloud Architect
   Match Score: 80.0%

2. DevOps Engineer
   Match Score: 73.4%

3. Cybersecurity Analyst
   Match Score: 10.7%



In [9]:
#User Input (Run this to input your own skills)

print("\n" + "=" * 50)
print("TRY YOUR OWN SKILLS")
print("=" * 50)

user_input = input("\nEnter your skills (space-separated): ")
recommend_job_roles(user_input)


TRY YOUR OWN SKILLS

📊 You entered skills: ['pytho', 'maths,', 'sql']

🎯 Top 3 Recommended Job Roles:

1. Data Scientist
   Match Score: 30.5%

2. Full Stack Developer
   Match Score: 25.6%

3. Backend Developer
   Match Score: 24.1%

